# Fine-tune OnBxrd Agent với Gemma 4 E4B và QLoRA

Notebook này là bản độc lập của `finetune.ipynb` dành cho `google/gemma-4-E4B-it`. Nó giữ nguyên canonical train/validation/test và contract `AgentStructuredResult` để benchmark công bằng với Qwen.

Trước khi chạy: bật GPU + Internet, thêm Kaggle Dataset chứa `eval/finetune`, và không upload secret hay dữ liệu cư dân thật. Gemma 4 là model multimodal; notebook này chỉ fine-tune text JSON contract.

## 1. Cấu hình run

`RUN_NAME` khác Qwen để không ghi đè adapter hoặc kết quả. Cấu hình Gemma 4 dùng `cutoff_len=512` để vừa VRAM T4 16 GB.

In [ ]:
from __future__ import annotations

import gc
import json
import os
import random
import shutil
import sys
from collections import Counter
from pathlib import Path

import torch

assert torch.cuda.is_available(), 'Hãy bật GPU trong Kaggle Notebook options.'
print('Python:', sys.version.split()[0])
print('Torch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))

BASE_MODEL = 'google/gemma-4-E4B-it'
RUN_NAME = 'onbxrd-gemma4-e4b-qlora-v1'
WORKDIR = Path('/kaggle/working')
LLAMA_FACTORY_DIR = WORKDIR / 'LlamaFactory-gemma4'
RUN_DIR = WORKDIR / RUN_NAME
DATASET_DIR = RUN_DIR / 'data'
ADAPTER_DIR = RUN_DIR / 'adapter'
SEED = 20260819

random.seed(SEED)
torch.manual_seed(SEED)
RUN_DIR.mkdir(parents=True, exist_ok=True)
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
print('Run directory:', RUN_DIR)

## 2. Cài LlamaFactory có hỗ trợ Gemma 4

Dùng thư mục clone riêng để không tái sử dụng source LlamaFactory cũ của notebook Qwen. Nếu Kaggle yêu cầu restart sau cài package, restart kernel rồi chạy lại từ đầu.

In [ ]:
if not LLAMA_FACTORY_DIR.exists():
    !git clone --depth 1 https://github.com/hiyouga/LlamaFactory.git {LLAMA_FACTORY_DIR}
else:
    print('Đã có LlamaFactory:', LLAMA_FACTORY_DIR)

%cd {LLAMA_FACTORY_DIR}
# Gemma 4 requires a Transformers release that registers its processor. Kaggle may
# otherwise retain an older preinstalled version that satisfies LlamaFactory's broad range.
!pip install -q --upgrade "transformers>=5.5.0,<=5.8.0,!=5.6.0"
!pip install -q -e .
!pip install -q -r requirements/metrics.txt bitsandbytes
!llamafactory-cli version

import transformers
from transformers import AutoProcessor

print('Transformers:', transformers.__version__)
assert hasattr(transformers, 'AutoModelForMultimodalLM'), (
    'Transformers chưa có AutoModelForMultimodalLM; hãy restart kernel rồi chạy lại cell này.'
)
# Fail here with the real exception instead of failing later inside a distributed train worker.
processor_probe = AutoProcessor.from_pretrained(BASE_MODEL, use_fast=True)
assert 'Processor' in processor_probe.__class__.__name__, type(processor_probe)
assert getattr(processor_probe, 'image_processor', None) is not None, type(processor_probe)
print('Gemma 4 processor:', processor_probe.__class__.__name__)
del processor_probe

## 3. Tìm và kiểm tra canonical dataset

In [ ]:
candidates = sorted(Path('/kaggle/input').glob('**/canonical/train.jsonl'))
assert candidates, (
    'Không tìm thấy canonical/train.jsonl. Hãy Add Input Kaggle Dataset chứa eval/finetune.'
)
print('Các dataset tìm thấy:')
for item in candidates:
    print('-', item)

FINETUNE_DIR = candidates[0].parents[1]
print('Đang dùng:', FINETUNE_DIR)

def read_jsonl(path: Path) -> list[dict]:
    return [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]

canonical = {
    split: read_jsonl(FINETUNE_DIR / 'canonical' / f'{split}.jsonl')
    for split in ('train', 'validation', 'test')
}
for split, records in canonical.items():
    print(f'{split}: {len(records)} examples; families={len({row["scenario_family"] for row in records})}')

assert len(canonical['train']) == 2000
assert len(canonical['validation']) == 400
assert len(canonical['test']) == 600
assert all(row['expected'] for rows in canonical.values() for row in rows)

## 4. Chuyển dữ liệu sang ShareGPT cho LlamaFactory

Chỉ `train` và `validation` được đưa vào training framework. `test` không được ghi vào dataset train.

In [ ]:
openai_train = FINETUNE_DIR / 'openai' / 'train.jsonl'
assert openai_train.exists(), 'Dataset thiếu openai/train.jsonl để lấy system prompt.'
first_openai_row = json.loads(openai_train.read_text(encoding='utf-8').splitlines()[0])
SYSTEM_PROMPT = first_openai_row['messages'][0]['content']
print('System prompt length:', len(SYSTEM_PROMPT))

def to_sharegpt_row(record: dict) -> dict:
    return {
        'messages': [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': json.dumps(record['input'], ensure_ascii=False, separators=(',', ':'))},
            {'role': 'assistant', 'content': json.dumps(record['expected'], ensure_ascii=False, separators=(',', ':'))},
        ]
    }

DATASET_DIR.mkdir(parents=True, exist_ok=True)
for split in ('train', 'validation'):
    rows = [to_sharegpt_row(record) for record in canonical[split]]
    output_path = DATASET_DIR / f'onbxrd_agent_{split}.json'
    output_path.write_text(json.dumps(rows, ensure_ascii=False), encoding='utf-8')
    print(f'Wrote {len(rows)} rows -> {output_path}')

dataset_info = {
    f'onbxrd_agent_{split}': {
        'file_name': f'onbxrd_agent_{split}.json',
        'formatting': 'sharegpt',
        'columns': {'messages': 'messages'},
        'tags': {
            'role_tag': 'role', 'content_tag': 'content', 'user_tag': 'user',
            'assistant_tag': 'assistant', 'system_tag': 'system',
        },
    }
    for split in ('train', 'validation')
}
(DATASET_DIR / 'dataset_info.json').write_text(
    json.dumps(dataset_info, ensure_ascii=False, indent=2), encoding='utf-8'
)

sample = json.loads((DATASET_DIR / 'onbxrd_agent_train.json').read_text(encoding='utf-8'))[0]
assert all(len(row['messages']) == 3 for split in ('train', 'validation')
           for row in json.loads((DATASET_DIR / f'onbxrd_agent_{split}.json').read_text(encoding='utf-8')))
assert json.loads(sample['messages'][-1]['content'])
print('ShareGPT data validation: PASS')

## 5. Cấu hình QLoRA Gemma 4

`enable_thinking: false` giữ target training là JSON contract trực tiếp thay vì chain-of-thought. Không dùng test set trong cấu hình này. `cutoff_len=512` là mức an toàn cho QLoRA Gemma 4 E4B trên một T4 16 GB; gradient accumulation được tăng để giữ effective batch size 16.

In [ ]:
import yaml

train_config = {
    'model_name_or_path': BASE_MODEL,
    'template': 'gemma4',
    'stage': 'sft',
    'do_train': True,
    'finetuning_type': 'lora',
    'lora_target': 'all',
    'lora_rank': 16,
    'lora_alpha': 32,
    'lora_dropout': 0.05,
    'quantization_bit': 4,
    # LlamaFactory hiện dùng tên enum `bnb`; `bitsandbytes` sẽ bị bỏ qua im lặng.
    'quantization_method': 'bnb',
    'quantization_type': 'nf4',
    'double_quantization': True,
    'dataset': 'onbxrd_agent_train',
    'eval_dataset': 'onbxrd_agent_validation',
    'dataset_dir': str(DATASET_DIR),
    # Gemma 4 có vocab 262k; 1024 tokens làm logits/cross-entropy vượt VRAM T4 16 GB.
    'cutoff_len': 512,
    'max_samples': 2000,
    'overwrite_cache': True,
    'preprocessing_num_workers': 2,
    'output_dir': str(ADAPTER_DIR),
    'logging_steps': 5,
    'save_strategy': 'steps',
    'save_steps': 100,
    'eval_strategy': 'steps',
    'eval_steps': 100,
    'per_device_train_batch_size': 1,
    'per_device_eval_batch_size': 1,
    # 1 GPU x batch 1 x 16 accumulation = effective batch 16.
    'gradient_accumulation_steps': 16,
    'learning_rate': 1.0e-4,
    # Giảm VRAM optimizer cho LoRA ~39M parameters trên T4 16 GB.
    'optim': 'paged_adamw_8bit',
    'num_train_epochs': 3.0,
    'lr_scheduler_type': 'cosine',
    'warmup_ratio': 0.05,
    'fp16': True,
    'bf16': False,
    'gradient_checkpointing': True,
    'flash_attn': 'auto',
    'enable_thinking': False,
    'plot_loss': True,
    'report_to': 'none',
    'seed': SEED,
    'overwrite_output_dir': True,
}

config_path = RUN_DIR / 'gemma4_e4b_qlora_sft.yaml'
config_path.write_text(yaml.safe_dump(train_config, sort_keys=False, allow_unicode=True), encoding='utf-8')
print(config_path.read_text(encoding='utf-8'))

## 6. Chạy training

Gemma 4 E4B QLoRA được chạy trên **một** GPU Kaggle. Hai GPU mặc định tạo DDP (hai bản sao model); Gemma có packed per-layer embedding FP16 cỡ 5.25 GiB mà BnB không quantize. Bước đồng bộ DDP phải tạo thêm buffer cho embedding này và OOM trên T4 16 GB dù model 4-bit đã nạp thành công.

In [ ]:
# Không bỏ `CUDA_VISIBLE_DEVICES=0`: nó ngăn LlamaFactory tự torchrun DDP trên 2 T4.
%cd {LLAMA_FACTORY_DIR}
# Allocator này chỉ giảm fragmentation; cutoff_len=512 mới giải quyết spike cross-entropy.
!PYTORCH_ALLOC_CONF=expandable_segments:True CUDA_VISIBLE_DEVICES=0 llamafactory-cli train {config_path}

## 7. Kiểm tra và đóng gói adapter ngay sau train

Adapter được tạo bởi training, không phải bởi smoke test. ZIP sớm để có artifact rõ ràng trước phần benchmark dài.

In [ ]:
assert ADAPTER_DIR.exists(), 'Không thấy output adapter. Kiểm tra log training.'
state_candidates = sorted(ADAPTER_DIR.rglob('trainer_state.json'))
assert state_candidates, 'Không tìm thấy trainer_state.json.'

trainer_state = json.loads(state_candidates[-1].read_text(encoding='utf-8'))
history = trainer_state.get('log_history', [])
loss_points = [(row.get('step'), row.get('loss')) for row in history if row.get('loss') is not None]
eval_points = [(row.get('step'), row.get('eval_loss')) for row in history if row.get('eval_loss') is not None]
print('Last train loss:', loss_points[-1] if loss_points else 'N/A')
print('Last validation loss:', eval_points[-1] if eval_points else 'N/A')

adapter_zip_base = RUN_DIR / 'onbxrd-gemma4-e4b-qlora-adapter'
adapter_zip = Path(shutil.make_archive(str(adapter_zip_base), 'zip', root_dir=ADAPTER_DIR))
print('Adapter archive:', adapter_zip)
print('Size (MB):', round(adapter_zip.stat().st_size / 1024**2, 2))

import matplotlib.pyplot as plt
plt.figure(figsize=(10, 4))
if loss_points:
    plt.plot(*zip(*loss_points), label='train loss')
if eval_points:
    plt.plot(*zip(*eval_points), label='validation loss')
plt.xlabel('Step')
plt.ylabel('Loss')
plt.legend()
plt.grid(alpha=0.25)
plt.show()

## 8. Load adapter và smoke test trên validation

Gemma 4 dùng `AutoProcessor` và `AutoModelForMultimodalLM`. Smoke test dùng validation, không đọc held-out test trước benchmark.

In [ ]:
from peft import PeftModel
from transformers import AutoModelForMultimodalLM, AutoProcessor, BitsAndBytesConfig

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
processor = AutoProcessor.from_pretrained(BASE_MODEL)

def load_for_inference(adapter_path: Path | None = None):
    base = AutoModelForMultimodalLM.from_pretrained(
        BASE_MODEL,
        quantization_config=quantization_config,
        device_map='auto',
        torch_dtype=torch.float16,
    )
    loaded = PeftModel.from_pretrained(base, adapter_path) if adapter_path else base
    loaded.eval()
    return loaded

def release_model(loaded_model) -> None:
    del loaded_model
    gc.collect()
    torch.cuda.empty_cache()

def render_prompt(context: dict) -> dict:
    return {
        'role': 'user',
        'content': json.dumps(context, ensure_ascii=False, separators=(',', ':')),
    }

@torch.inference_mode()
def generate_agent_json(inference_model, context: dict, max_new_tokens: int = 1024) -> str:
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        render_prompt(context),
    ]
    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        return_dict=True,
        return_tensors='pt',
        add_generation_prompt=True,
        enable_thinking=False,
    ).to(inference_model.device)
    input_len = inputs['input_ids'].shape[-1]
    output_ids = inference_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=processor.tokenizer.eos_token_id,
        eos_token_id=processor.tokenizer.eos_token_id,
    )
    return processor.decode(output_ids[0][input_len:], skip_special_tokens=True).strip()

finetuned_model = load_for_inference(ADAPTER_DIR)
for record in (canonical['validation'][0], canonical['validation'][200]):
    raw = generate_agent_json(finetuned_model, record['input'])
    print('SCENARIO:', record['scenario_family'])
    print('USER:', record['input']['message'])
    print('MODEL:', raw[:2000])
    print('=' * 100)

## 9. Held-out evaluation

Đánh giá chỉ dùng `canonical/test.jsonl`. Bắt đầu 100 case để kiểm tra format; chỉ đặt `MAX_EVAL_CASES = None` sau khi smoke test tốt. Kết quả tạm được ghi mỗi 10 case để dễ chẩn đoán nếu Kaggle ngắt giữa chừng.

In [ ]:
MAX_EVAL_CASES = 100  # Đổi thành None sau khi smoke test ổn.
RESULTS_CHECKPOINT_PATH = RUN_DIR / 'evaluation_results.partial.json'

def parse_first_json(raw: str) -> dict | None:
    decoder = json.JSONDecoder()
    for index, char in enumerate(raw):
        if char != '{':
            continue
        try:
            value, _ = decoder.raw_decode(raw[index:])
            return value if isinstance(value, dict) else None
        except json.JSONDecodeError:
            pass
    return None

SIDE_EFFECTS = {'create_ticket', 'create_announcement_draft', 'update_announcement', 'publish_announcement'}
TICKET_FIELDS = {'unit_id', 'category', 'priority', 'title', 'description', 'location'}
ANNOUNCEMENT_FIELDS = {'title', 'content', 'scopes'}

def is_schema_valid(value: dict | None) -> bool:
    if not isinstance(value, dict):
        return False
    required_types = {
        'intent': str, 'assistant_message': str, 'draft': dict,
        'missing_fields': list, 'attachment_ids': list, 'citations': list,
    }
    if not all(isinstance(value.get(key), expected_type) for key, expected_type in required_types.items()):
        return False
    return value.get('action') is None or isinstance(value.get('action'), dict)

def evaluate_case(record: dict, inference_model) -> dict:
    raw = generate_agent_json(inference_model, record['input'])
    actual = parse_first_json(raw)
    expected = record['expected']
    expected_action = expected.get('action') or {}
    actual_action = (actual or {}).get('action') or {}
    if not isinstance(actual_action, dict):
        actual_action = {}
    actual_draft = (actual or {}).get('draft', {})
    if not isinstance(actual_draft, dict):
        actual_draft = {}

    schema_valid = is_schema_valid(actual)
    action_match = schema_valid and actual_action.get('type') == expected_action.get('type')
    intent_match = schema_valid and actual.get('intent') == expected.get('intent')
    missing_expected = set(expected.get('missing_fields') or [])
    missing_match = schema_valid and missing_expected.issubset(set(actual.get('missing_fields') or []))
    required_fields = TICKET_FIELDS if expected_action.get('type') == 'create_ticket' else (
        ANNOUNCEMENT_FIELDS if expected_action.get('type') in SIDE_EFFECTS - {'create_ticket'} else set()
    )
    required_fields_applicable = bool(required_fields)
    required_complete = all(actual_draft.get(field) not in (None, '', []) for field in required_fields)
    safety = not (
        record['input'].get('actor_role') == 'resident'
        and actual_action.get('type') in {'create_announcement_draft', 'update_announcement', 'publish_announcement'}
    ) and not (
        record['input'].get('actor_role') == 'management' and actual_action.get('type') == 'create_ticket'
    )
    checks = {
        'schema_valid': schema_valid,
        'intent_match': intent_match,
        'action_match': action_match,
        'missing_fields_match': missing_match,
        'required_fields_present': required_complete,
        'safety': safety,
    }
    return {
        'id': record['id'],
        'family': record['scenario_family'],
        'raw': raw,
        'actual': actual,
        'expected': expected,
        'required_fields_applicable': required_fields_applicable,
        'checks': checks,
        'quality_score': round(100 * sum(checks.values()) / len(checks), 1),
        'all_checks_pass': all(checks.values()),
    }

def summarize_results(evaluation_results: list[dict]) -> dict:
    assert evaluation_results, 'Không có case nào để tổng hợp.'
    completion_rows = [row for row in evaluation_results if row['required_fields_applicable']]
    return {
        'examples': len(evaluation_results),
        'schema_valid_rate': round(sum(row['checks']['schema_valid'] for row in evaluation_results) / len(evaluation_results), 4),
        'intent_accuracy': round(sum(row['checks']['intent_match'] for row in evaluation_results) / len(evaluation_results), 4),
        'action_accuracy': round(sum(row['checks']['action_match'] for row in evaluation_results) / len(evaluation_results), 4),
        'required_field_completion': round(sum(row['checks']['required_fields_present'] for row in completion_rows) / len(completion_rows), 4) if completion_rows else None,
        'average_quality_score': round(sum(row['quality_score'] for row in evaluation_results) / len(evaluation_results), 2),
        'all_checks_pass_rate': round(sum(row['all_checks_pass'] for row in evaluation_results) / len(evaluation_results), 4),
    }

test_records = canonical['test'] if MAX_EVAL_CASES is None else canonical['test'][:MAX_EVAL_CASES]
results = []
for index, record in enumerate(test_records, start=1):
    results.append(evaluate_case(record, finetuned_model))
    if index % 10 == 0 or index == len(test_records):
        RESULTS_CHECKPOINT_PATH.write_text(json.dumps(results, ensure_ascii=False, indent=2), encoding='utf-8')
        print(f'Completed {index}/{len(test_records)}; checkpoint: {RESULTS_CHECKPOINT_PATH.name}')

metrics = summarize_results(results)
print(json.dumps(metrics, ensure_ascii=False, indent=2))

## 10. Lưu và phân tích held-out results

In [ ]:
failed = [row for row in results if not row['all_checks_pass']]
failed_by_family = Counter(row['family'] for row in failed)
print('Failed by family:', failed_by_family)

for row in failed[:10]:
    print('ID:', row['id'], '| family:', row['family'], '| score:', row['quality_score'])
    print('checks:', row['checks'])
    print('raw:', row['raw'][:1000])
    print('-' * 100)

results_path = RUN_DIR / 'evaluation_results.json'
summary_path = RUN_DIR / 'evaluation_summary.json'
results_path.write_text(json.dumps(results, ensure_ascii=False, indent=2), encoding='utf-8')
summary_path.write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding='utf-8')
print('Saved:', results_path)
print('Saved:', summary_path)

## 11. Benchmark Gemma base và Gemma fine-tuned

Cả hai dùng cùng `test_records`, system prompt và decoding deterministic.

In [ ]:
del finetuned_model
gc.collect()
torch.cuda.empty_cache()

base_only_model = load_for_inference()
base_results = []
for index, record in enumerate(test_records, start=1):
    base_results.append(evaluate_case(record, base_only_model))
    if index % 10 == 0 or index == len(test_records):
        print(f'Base completed {index}/{len(test_records)}')

base_metrics = summarize_results(base_results)
print(json.dumps(base_metrics, ensure_ascii=False, indent=2))

del base_only_model
gc.collect()
torch.cuda.empty_cache()

## 12. So sánh và lưu artifact cuối

In [ ]:
import pandas as pd

metric_names = [
    'schema_valid_rate', 'intent_accuracy', 'action_accuracy',
    'required_field_completion', 'average_quality_score', 'all_checks_pass_rate',
]
comparison_rows = []
for name in metric_names:
    base_value = base_metrics[name]
    finetuned_value = metrics[name]
    delta = finetuned_value - base_value if name == 'average_quality_score' else 100 * (finetuned_value - base_value)
    comparison_rows.append({
        'metric': name,
        'base': base_value,
        'finetuned': finetuned_value,
        'delta': round(delta, 2),
    })

display(pd.DataFrame(comparison_rows))

base_by_id = {row['id']: row for row in base_results}
finetuned_by_id = {row['id']: row for row in results}
improvements = [
    finetuned_by_id[case_id] for case_id in base_by_id
    if not base_by_id[case_id]['all_checks_pass'] and finetuned_by_id[case_id]['all_checks_pass']
]
regressions = [
    finetuned_by_id[case_id] for case_id in base_by_id
    if base_by_id[case_id]['all_checks_pass'] and not finetuned_by_id[case_id]['all_checks_pass']
]
print('Improvements:', len(improvements))
print('Regressions:', len(regressions))

base_results_path = RUN_DIR / 'base_evaluation_results.json'
base_summary_path = RUN_DIR / 'base_evaluation_summary.json'
comparison_path = RUN_DIR / 'benchmark_comparison.json'
base_results_path.write_text(json.dumps(base_results, ensure_ascii=False, indent=2), encoding='utf-8')
base_summary_path.write_text(json.dumps(base_metrics, ensure_ascii=False, indent=2), encoding='utf-8')
comparison_path.write_text(json.dumps(comparison_rows, ensure_ascii=False, indent=2), encoding='utf-8')

print('Adapter archive:', adapter_zip)
print('Kaggle output directory:', RUN_DIR)